In [15]:
# Cell 1 — Imports
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import os
os.environ["WDM_SSL_VERIFY"] = "0"

import sys
try:
    sys.stdout.reconfigure(encoding="utf-8")
except AttributeError:
    pass

import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.microsoft import EdgeChromiumDriverManager


def log(message: str) -> None:
    print(message, flush=True)

In [2]:
# Cell 2 — Configuration
URL = "https://compass.dialecticanet.com/client-view/h5p_z5GiE5TPgUT_JCO_C0GrsuLAU6MG5grwA4lVDktNQ-T-0ltAdFHDMw4T8FjYvBTB6zeq-gBDO2hKhlxDZQ-4d12fb969270/presentation?search=&tab=longlist"

In [16]:
# ============================================
# Cell 3 — Browser setup
# ============================================

def create_driver():

    log("Preparing Edge driver...")

    options = webdriver.EdgeOptions()

    options.add_argument("--start-maximized")
    options.add_argument("--ignore-certificate-errors")

    # Stability fixes
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--remote-debugging-port=9222")

    # Dedicated Selenium profile
    options.add_argument(
        r"--user-data-dir=C:\Users\ND126HF\selenium_edge_profile"
    )

    driver_path = EdgeChromiumDriverManager().install()

    log(f"Using Edge driver: {driver_path}")

    driver = webdriver.Edge(
        service=Service(driver_path),
        options=options
    )

    driver.set_page_load_timeout(120)
    driver.set_script_timeout(120)

    return driver

In [17]:
# ============================================
# Cell 4 — Manual login helper
# ============================================

def open_and_wait_for_manual_login(driver, url: str):

    log("Opening Dialectica page...")

    driver.get(url)

    input(
        "\nComplete Dialectica login manually in the Edge window.\n"
        "After the longlist page fully loads, press Enter here to continue..."
    )

    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )

    log("Manual login complete. Continuing scrape...")

In [18]:
# ============================================
# Cell 5 — Helper functions
# ============================================

def clean_text(value: str) -> str:
    return re.sub(r"\s+", " ", value or "").strip()


def looks_like_name(line: str) -> bool:

    if not line:
        return False

    reject_words = [
        "Dialectica",
        "Compass",
        "Longlist",
        "Shortlist",
        "Search",
        "Presentation",
        "Client",
        "Project",
        "Filter",
        "Sort",
        "Availability",
        "Rate",
        "Location",
        "Profile"
    ]

    if any(w.lower() in line.lower() for w in reject_words):
        return False

    words = line.split()

    if not 2 <= len(words) <= 4:
        return False

    return all(
        re.match(r"^[A-ZÀ-ÿ][A-Za-zÀ-ÿ'’.-]+$", w)
        for w in words
    )


def extract_screener(lines: list) -> list:

    screener = []

    stop_words = [
        "work experience",
        "employment history",
        "professional experience",
        "contact expert",
        "request call",
        "schedule",
        "shortlist",
        "remove",
        "back",
        "close"
    ]

    for i, line in enumerate(lines):

        if re.match(r"^\d+[\.\)]\s+", line):

            question = line
            answer = "No answer"

            if i + 1 < len(lines):

                next_line = lines[i + 1]

                if (
                    not re.match(r"^\d+[\.\)]\s+", next_line)
                    and not any(w in next_line.lower() for w in stop_words)
                ):
                    answer = next_line

            screener.append({
                "question": question,
                "answer": answer
            })

    return screener


def parse_role_company(lines: list, name_idx: int) -> tuple:

    nearby = lines[
        max(0, name_idx - 4):
        min(len(lines), name_idx + 8)
    ]

    for line in nearby:

        if " at " in line.lower():

            parts = re.split(
                r"\s+at\s+",
                line,
                flags=re.IGNORECASE
            )

            if len(parts) >= 2:
                return (
                    clean_text(parts[0]),
                    clean_text(parts[1])
                )

        if " - " in line:

            parts = line.split(" - ")

            if len(parts) >= 2:
                return (
                    clean_text(parts[1]),
                    clean_text(parts[0])
                )

        if " | " in line:

            parts = line.split(" | ")

            if len(parts) >= 2:
                return (
                    clean_text(parts[0]),
                    clean_text(parts[1])
                )

    return "", ""

In [19]:
# ============================================
# Cell 6 — Main scraper
# ============================================

def fetch_all_experts(url: str) -> list:

    driver = None
    experts = []

    try:

        driver = create_driver()

        open_and_wait_for_manual_login(driver, url)

        log("Page loaded. Waiting for longlist...")
        time.sleep(5)

        # Scroll to load all experts
        previous_height = 0

        for scroll_count in range(30):

            driver.execute_script(
                "window.scrollTo(0, document.body.scrollHeight);"
            )

            time.sleep(1.5)

            current_height = driver.execute_script(
                "return document.body.scrollHeight"
            )

            if current_height == previous_height:
                break

            previous_height = current_height

            log(f"Scrolled page ({scroll_count + 1})")

        # Capture page text
        pre_click_lines = driver.find_element(
            By.TAG_NAME,
            "body"
        ).text.split("\n")

        pre_click_lines = [
            clean_text(l)
            for l in pre_click_lines
            if clean_text(l)
        ]

        # Save diagnostic dump
        with open(
            "dialectica_page_dump.txt",
            "w",
            encoding="utf-8"
        ) as f:

            for idx, line in enumerate(pre_click_lines):
                f.write(f"{idx}: {line}\n")

        log("Saved diagnostic dump: dialectica_page_dump.txt")

        # Find candidate expert names
        candidate_names = []

        for line in pre_click_lines:

            if (
                looks_like_name(line)
                and line not in candidate_names
            ):
                candidate_names.append(line)

        log(f"Candidate experts found: {len(candidate_names)}")

        # Extract visible list-view data
        pre_click_data = {}

        for name in candidate_names:

            idx = pre_click_lines.index(name)

            nearby = pre_click_lines[idx: idx + 12]

            geography = ""
            segment = ""
            rate = ""
            availability = ""

            for line in nearby:

                lower = line.lower()

                if any(
                    x in lower
                    for x in [
                        "available",
                        "availability",
                        "next week",
                        "this week"
                    ]
                ):
                    availability = line

                elif any(
                    currency in line
                    for currency in [
                        "€",
                        "$",
                        "£",
                        "EUR",
                        "USD",
                        "GBP"
                    ]
                ):
                    rate = line

                elif any(
                    x in lower
                    for x in [
                        "united kingdom",
                        "uk",
                        "germany",
                        "france",
                        "spain",
                        "italy",
                        "netherlands",
                        "europe",
                        "usa",
                        "us"
                    ]
                ):
                    geography = line

                elif (
                    not segment
                    and line != name
                    and len(line) < 80
                ):
                    segment = line

            pre_click_data[name] = {
                "geography": geography,
                "segment": segment,
                "rate": rate,
                "availability": availability
            }

        # Open each expert profile
        for i, name in enumerate(pre_click_data.keys()):

            try:

                log(
                    f"Opening expert "
                    f"{i + 1}/{len(pre_click_data)}: {name}"
                )

                expert_el = driver.find_element(
                    By.XPATH,
                    f"//*[normalize-space(text())='{name}']"
                )

                driver.execute_script(
                    """
                    arguments[0].scrollIntoView(true);
                    window.scrollBy(0, -150);
                    """,
                    expert_el
                )

                time.sleep(1)

                driver.execute_script(
                    "arguments[0].click();",
                    expert_el
                )

                time.sleep(3)

                post_lines = driver.find_element(
                    By.TAG_NAME,
                    "body"
                ).text.split("\n")

                post_lines = [
                    clean_text(l)
                    for l in post_lines
                    if clean_text(l)
                ]

                name_idx = next(
                    (
                        idx
                        for idx, l in enumerate(post_lines)
                        if l == name
                    ),
                    None
                )

                if name_idx is None:

                    log(
                        f"Skipping {name} — "
                        f"could not locate expanded profile"
                    )

                    continue

                role, company = parse_role_company(
                    post_lines,
                    name_idx
                )

                screener = extract_screener(post_lines)

                entry = pre_click_data[name]

                experts.append({
                    "name": name,
                    "company": company,
                    "role": role,
                    "geography": entry["geography"],
                    "segment": entry["segment"],
                    "rate": entry["rate"],
                    "availability": entry["availability"],
                    "screener_responses": screener
                })

                log(f"Extracted: {name}")

                # Attempt to close modal/profile
                try:

                    close_buttons = driver.find_elements(
                        By.XPATH,
                        """
                        //button[
                            contains(., 'Close')
                            or contains(., 'Back')
                            or @aria-label='Close'
                        ]
                        """
                    )

                    if close_buttons:

                        driver.execute_script(
                            "arguments[0].click();",
                            close_buttons[0]
                        )

                        time.sleep(1)

                except:
                    pass

            except Exception as e:

                log(
                    f"Error on expert {i + 1} "
                    f"({name}): {e}"
                )

                continue

    except Exception as e:

        log(f"Fatal error: {e}")

    finally:

        if driver is not None:
            driver.quit()

    return experts

In [20]:
# ============================================
# Cell 7 — Display as table
# ============================================

def display_as_table(experts: list):

    rows = []

    for e in experts:

        screener_text = "\n".join(
            [
                f"Q: {qa['question']}\nA: {qa['answer']}"
                for qa in e["screener_responses"]
            ]
        ) if e["screener_responses"] else "None"

        rows.append({
            "Name": e["name"],
            "Geography": e["geography"],
            "Segment": e["segment"],
            "Rate": e["rate"],
            "Role": e["role"],
            "Company": e["company"],
            "Screener Responses": screener_text,
            "Availability": e["availability"]
        })

    df = pd.DataFrame(rows)

    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.max_rows", None)

    return df

In [21]:
# ============================================
# Cell 8 — Run scraper
# ============================================

def main():

    log("Starting Dialectica scrape...")

    experts = fetch_all_experts(URL)

    df = display_as_table(experts)

    log(f"Rows scraped: {len(df)}")

    if df.empty:

        log(
            "No rows were scraped.\n"
            "Check dialectica_page_dump.txt"
        )

    else:

        df.to_excel(
            "dialectica_experts.xlsx",
            index=False
        )

        df.to_csv(
            "dialectica_experts.csv",
            index=False,
            encoding="utf-8-sig"
        )

        log(
            "Saved outputs:\n"
            "- dialectica_experts.xlsx\n"
            "- dialectica_experts.csv"
        )

    return df


df = main()
df

Starting Dialectica scrape...
Preparing Edge driver...
Using Edge driver: C:\Users\ND126HF\.wdm\drivers\edgedriver\win64\148.0.3967.70\msedgedriver.exe
Opening Dialectica page...
Manual login complete. Continuing scrape...
Page loaded. Waiting for longlist...
Scrolled page (1)
Saved diagnostic dump: dialectica_page_dump.txt
Candidate experts found: 14
Opening expert 1/14: Konstantinos Michail Servis
Extracted: Konstantinos Michail Servis
Opening expert 2/14: Customer Success Specialist
Error on expert 2 (Customer Success Specialist): Message: no such element: Unable to locate element: {"method":"xpath","selector":"//*[normalize-space(text())='Customer Success Specialist']"}
  (Session info: MicrosoftEdge=148.0.3967.54); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	msedgedriver!GetHandleVerifier [0x7ff62892e295+e0f5]
	msedgedriver!GetHandleVerifier [0x7ff62892e2f4+e154]
	msedged

,Name,Geography,Segment,Rate,Role,Company,Screener Responses,Availability
0,Konstantinos Michail Servis,,konstantinos.servis@dialecticanet.com,,,,None,
1,CoreNet Global Benelux Chapter,#73 Markus Laub,Chair/board member,,Now,Jan '23,None,
2,NORDSEE GmbH,,Operational Excellence Manager,,Mar '26,Jun '25,"Q: 1) Role & involvement\nA: I am involved in the day-to-day operational management and optimization of IWMS and related enterprise systems, including Axxerion for facilities and maintenance workflows. I also support process alignment, data consistency, and coordination between operations, finance, and technical teams.\nQ: 2) Type of solution used\nA: We use a combination of integrated and best-of-breed solutions across IWMS, workforce management, ERP, and analytics. Key tools include Axxerion for facility/asset management, E2N for workforce scheduling, SAP/Oracle for ERP processes, and Power BI for reporting.",
3,Crypto Valley Association,Europe,Nov '24 - Now,,Now,Nov '24,None,
4,FC Twente,,ICT Sourcing,,Now,Jan '21,None,
5,Waser Works AG,Terms of Use,"Mitinhaber und Verwaltungsrat / Co-owner, Member of the board of directors",,Now,Jun '18,None,
